In [ ]:
import pandas as pd
from config.config import *
import numpy as np
import statsmodels.api as sm

In [22]:
df = pd.read_feather('data/month_label.feather')
index_df = pd.read_feather('data/month_HS300_rf.feather')

In [23]:
df

nav_date  accum_nav    adj_nav     fd_share     label  \
ann_date   ts_code                                                            
2016-04-30 000011.OF  20160429    13.9370  17.213300   18642.5278  0.002014   
           000309.OF  20160429     1.5390   1.539000   32296.7705  0.035666   
           000409.OF  20160429     1.4010   1.401000   36018.4498  0.007914   
           000471.OF  20160429     2.2620   2.262000  177241.4719 -0.013089   
           000513.OF  20160429     1.3900   1.390000   74609.3099 -0.024561   
...                        ...        ...        ...          ...       ...   
2026-03-31 540008.OF  20260330     2.9249   3.079717  155482.9886 -0.069103   
           540009.OF  20260330     1.7303   1.714082   24682.4579 -0.082504   
           540010.OF  20260330     4.1115   4.111500   16997.8038  0.015913   
           550008.OF  20260330     2.9682   3.523751  132721.6502 -0.050345   
           960000.OF  20260330     2.2753   2.275300    3922.2942 -0.056049   

                        weight  
ann_date   ts_code              
2016-04-30 000011.OF  0.000767  
           000309.OF  0.001328  
           000409.OF  0.001481  
           000471.OF  0.007290  
           000513.OF  0.003069  
...                        ...  
2026-03-31 540008.OF  0.013204  
           540009.OF  0.002096  
           540010.OF  0.001444  
           550008.OF  0.011271  
           960000.OF  0.000333  

[21716 rows x 6 columns]

In [24]:
index_df

,close,yield,excess
trade_date,,,
2016-04-30,-0.019062,0.011,-0.030062
2016-05-31,0.004059,0.011,-0.006941
2016-06-30,-0.004934,0.011,-0.015934
2016-07-31,0.015856,0.011,0.004856
2016-08-31,0.038660,0.011,0.027660
...,...,...,...
2025-11-30,-0.024567,0.011,-0.035567
2025-12-31,0.022815,0.011,0.011815
2026-01-31,0.016501,0.011,0.005501


In [25]:
def calc_fund_rolling_alpha(fund_df, market_df, window=12):
    """
    对单只基金计算滚动12个月CAPM Alpha
    fund_df: 单只基金的df (ann_date索引, label列)
    market_df: 指数数据 (trade_date索引, close,yield列)
    """
    # 排序确保时间顺序
    fund_df = fund_df.sort_index()
    
    # 提取基金时间序列 + 对齐市场数据
    dates = fund_df.index.get_level_values('ann_date')
    fund_ret = fund_df['label']  # 基金收益率
    risk_free = market_df.loc[dates, 'yield']  # 无风险利率
    market_ret = market_df.loc[dates, 'close']  # 沪深300收益率
    
    # 计算 基金超额收益 = 基金收益 - 无风险收益
    fund_excess = fund_ret - risk_free.values
    # 计算 市场超额收益 = 沪深300收益 - 无风险收益
    market_excess = market_ret.values - risk_free.values
    
    # 构建带时间的Series
    fund_excess = pd.Series(fund_excess, index=dates)
    market_excess = pd.Series(market_excess, index=dates)
    
    # 滚动12个月回归计算Alpha
    alpha_series = pd.Series(index=dates, dtype=np.float64)
    
    # 从第window个时间开始滚动计算
    for i in range(window, len(dates)):
        # 过去12个月窗口
        window_dates = dates[i-window:i]
        y = fund_excess[window_dates]  # 基金超额收益
        x = market_excess[window_dates]  # 市场超额收益
        
        # 剔除缺失值
        valid = ~(np.isnan(y) | np.isnan(x))
        if valid.sum() < 6:  # 有效数据不足则跳过
            continue
            
        y = y[valid]
        x = x[valid]
        
        # CAPM回归：添加截距项
        x_const = sm.add_constant(x)
        model = sm.OLS(y, x_const).fit()
        
        # 保存截距项 alpha 到当期时间
        alpha_series.iloc[i] = model.params.iloc[0]
    
    return alpha_series

In [26]:
from tqdm import tqdm   # 进度条，AI加的，感觉很有趣
# 重置索引方便分组
df_calc = df.reset_index()
# 存储所有基金的Alpha结果
alpha_list = []

# 按基金分组计算（带进度条）
for ts_code, group in tqdm(df_calc.groupby('ts_code'), desc="计算基金Alpha"):
    # 构建单只基金的时间序列DF
    fund_df = group.set_index('ann_date')[['label']].sort_index()
    # 计算滚动Alpha
    alpha = calc_fund_rolling_alpha(fund_df, index_df)
    # 保存结果
    alpha_res = pd.DataFrame({
        'ann_date': alpha.index,
        'ts_code': ts_code,
        'alpha': alpha.values
    })
    alpha_list.append(alpha_res)

# 合并所有基金Alpha
alpha_df = pd.concat(alpha_list, ignore_index=True)

计算基金Alpha: 100%|██████████| 181/181 [00:14<00:00, 12.08it/s]


In [27]:
alpha_df

,ann_date,ts_code,alpha
0,2016-04-30,000011.OF,NaN
1,2016-05-31,000011.OF,NaN
2,2016-06-30,000011.OF,NaN
3,2016-07-31,000011.OF,NaN
4,2016-08-31,000011.OF,NaN
...,...,...,...
21711,2025-11-30,960000.OF,0.007179
21712,2025-12-31,960000.OF,0.009962
21713,2026-01-31,960000.OF,0.011091
21714,2026-02-28,960000.OF,0.014034


In [28]:
# 合并Alpha到原数据
result_df = df_calc.merge(alpha_df, on=['ann_date', 'ts_code'], how='left')

# 剔除无Alpha的早期数据（这就是top25_final缺少最早一年数据的原因）
result_valid = result_df.dropna(subset=['alpha']).copy()

# 按月分组：计算75分位数（前25%阈值）
def filter_top25(group):
    threshold = group['alpha'].quantile(0.75)
    return group[group['alpha'] >= threshold]

top25_df = result_valid.groupby('ann_date', group_keys=False).apply(filter_top25)   # 这里就把每个月分的明明白白

top25_df2 = top25_df[['ann_date', 'ts_code','adj_nav', 'fd_share', 'label']].set_index(['ann_date', 'ts_code'])


/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57325/171405646.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top25_df = result_valid.groupby('ann_date', group_keys=False).apply(filter_top25)   # 这里就把每个月分的明明白白


In [29]:
top25_df2['total_fd_share'] = top25_df2.groupby('ann_date')['fd_share'].transform('sum')
top25_df2['weight'] = top25_df2['fd_share'] / top25_df2['total_fd_share']
top25_df2 = top25_df2.drop(columns=['total_fd_share', 'fd_share'])
top25_df2

adj_nav     label    weight
ann_date   ts_code                                 
2017-04-30 000309.OF   1.836000 -0.024442  0.022620
           000409.OF   1.670000  0.006024  0.007996
           000577.OF   2.384000  0.027586  0.020153
           000628.OF   1.340000  0.068581  0.002976
           000751.OF   1.633000  0.034854  0.003568
...                         ...       ...       ...
2026-03-31 460007.OF   4.417000  0.056951  0.001926
           519003.OF  10.672724 -0.036148  0.019410
           519606.OF   3.408800 -0.079829  0.006246
           540007.OF   3.503321 -0.141499  0.010052
           540010.OF   4.111500  0.015913  0.005392

[4967 rows x 3 columns]

In [30]:
top25_df2.to_feather('data/top25_month_label.feather')